In [70]:
import pandas as pd
import numpy as np 
import requests
import zipfile
import io
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [71]:
data = pd.read_csv("./bank+marketing/bank/bank-full.csv", sep=';')
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [72]:
data.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [73]:
# age,
# job,
# marital,
# education,
# balance,
# housing,
# contact,
# day,
# month,
# duration,
# campaign,
# pdays,
# previous,
# poutcome,
# y

# to exclude 'default', 'loan'

In [74]:
df = data.drop(columns=['default', 'loan']).copy()
df.shape

(45211, 15)

In [75]:
df.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,2143,yes,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,29,yes,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,2,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,1506,yes,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,1,no,unknown,5,may,198,1,-1,0,unknown,no


In [76]:
df.y.value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [77]:
df.dtypes

age           int64
job          object
marital      object
education    object
balance       int64
housing      object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

In [78]:
df.columns

Index(['age', 'job', 'marital', 'education', 'balance', 'housing', 'contact',
       'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'y'],
      dtype='object')

In [98]:
categorical = ['job', 'marital', 'education',  'housing', 'contact', 'month', 'poutcome']
numerical = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

   Input variables:
   # bank client data:
  - 1 - age (numeric)
  - 2 - job : type of job (categorical: "admin.","unknown","unemployed","management","housemaid","entrepreneur","student",
                                       "blue-collar","self-employed","retired","technician","services") 
  - 3 - marital : marital status (categorical: "married","divorced","single"; note: "divorced" means divorced or widowed)
  - 4 - education (categorical: "unknown","secondary","primary","tertiary")
  - 5 - default: has credit in default? (binary: "yes","no")
  - 6 - balance: average yearly balance, in euros (numeric) 
  - 7 - housing: has housing loan? (binary: "yes","no")
  - 8 - loan: has personal loan? (binary: "yes","no")
   # related with the last contact of the current campaign:
  - 9 - contact: contact communication type (categorical: "unknown","telephone","cellular") 
  - 10 - day: last contact day of the month (numeric)
  - 11 - month: last contact month of year (categorical: "jan", "feb", "mar", ..., "nov", "dec")
  - 12 - duration: last contact duration, in seconds (numeric)
   # other attributes:
  - 13 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
  - 14 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)
  - 15 - previous: number of contacts performed before this campaign and for this client (numeric)
  - 16 - poutcome: outcome of the previous marketing campaign (categorical: "unknown","other","failure","success")

  Output variable (desired target):
  - 17 - y - has the client subscribed a term deposit? (binary: "yes","no")

In [80]:
df.isnull().any().sum()

0

## Question 1

In [81]:
df.education.mode()[0]

'secondary'

In [82]:
df.education.value_counts()

education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

In [83]:
unique_features = list(df.education.unique())
unique_features

['tertiary', 'secondary', 'unknown', 'primary']

In [84]:
for f in unique_features:
    print(f, df[df['education'] == f].count()[0])

tertiary 13301
secondary 23202
unknown 1857
primary 6851


C:\Users\MUHAMMED IBRAHIM\AppData\Local\Temp\ipykernel_19388\1239316148.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f, df[df['education'] == f].count()[0])
C:\Users\MUHAMMED IBRAHIM\AppData\Local\Temp\ipykernel_19388\1239316148.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f, df[df['education'] == f].count()[0])
C:\Users\MUHAMMED IBRAHIM\AppData\Local\Temp\ipykernel_19388\1239316148.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by posi

## Question 2

In [85]:
df['age'].corr(df['balance'])

0.09778273937134742

In [86]:
df['day'].corr(df['campaign'])

0.1624902163261929

In [87]:
df['day'].corr(df['pdays'])

-0.09304407377294044

In [88]:
# the biggest correlation: pdays and previous
df['pdays'].corr(df['previous'])

0.4548196354805016

### Target encoding

In [89]:
(df['y'] == 'yes').astype(int).value_counts()

y
0    39922
1     5289
Name: count, dtype: int64

In [90]:
df['y'] = (df['y'] == 'yes').astype(int)

## Split of the data

In [91]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [92]:
len(df_train), len(df_val), len(df_test)

(27126, 9042, 9043)

In [93]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.y.values
y_val = df_val.y.values
y_test = df_test.y.values

# del df_full_train['y']
# del df_train['y']
# del df_val['y']
# del df_test['y']

## Question 3

In [94]:
from sklearn.metrics import mutual_info_score

def mutual_info_subscribtion_score(series):
    return round(mutual_info_score(series, df_train.y), 2)
 
mi = df_train[categorical].apply(mutual_info_subscribtion_score)
mi

job          0.01
marital      0.00
education    0.00
housing      0.01
contact      0.01
poutcome     0.03
dtype: float64

In [97]:
mi.sort_values(ascending=False)

poutcome     0.03
job          0.01
housing      0.01
contact      0.01
marital      0.00
education    0.00
dtype: float64

## Question 4

In [100]:
del df_full_train['y']
del df_train['y']
del df_val['y']
del df_test['y']

In [134]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')
val_dict = df_val[categorical + numerical].to_dict(orient='records')
train_dict[0]

{'job': 'technician',
 'marital': 'single',
 'education': 'tertiary',
 'housing': 'yes',
 'contact': 'cellular',
 'month': 'aug',
 'poutcome': 'unknown',
 'age': 32,
 'balance': 1100,
 'day': 11,
 'duration': 67,
 'campaign': 1,
 'pdays': -1,
 'previous': 0}

In [135]:
from sklearn.feature_extraction import DictVectorizer
dv_v = DictVectorizer(sparse=False)

dv_v.fit(train_dict)

dv_v.get_feature_names_out()

array(['age', 'balance', 'campaign', 'contact=cellular',
       'contact=telephone', 'contact=unknown', 'day', 'duration',
       'education=primary', 'education=secondary', 'education=tertiary',
       'education=unknown', 'housing=no', 'housing=yes', 'job=admin.',
       'job=blue-collar', 'job=entrepreneur', 'job=housemaid',
       'job=management', 'job=retired', 'job=self-employed',
       'job=services', 'job=student', 'job=technician', 'job=unemployed',
       'job=unknown', 'marital=divorced', 'marital=married',
       'marital=single', 'month=apr', 'month=aug', 'month=dec',
       'month=feb', 'month=jan', 'month=jul', 'month=jun', 'month=mar',
       'month=may', 'month=nov', 'month=oct', 'month=sep', 'pdays',
       'poutcome=failure', 'poutcome=other', 'poutcome=success',
       'poutcome=unknown', 'previous'], dtype=object)

In [136]:
X_train = dv_v.transform(train_dict)
X_val = dv_v.transform(val_dict)
X_val.shape

(9042, 47)

In [137]:
from sklearn.linear_model import LogisticRegression
 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [138]:
training_result_prob = model.predict_proba(X_train)

In [139]:
training_result = model.predict(X_train)

In [140]:
training_result_prob

array([[0.96836658, 0.03163342],
       [0.96723941, 0.03276059],
       [0.93624865, 0.06375135],
       ...,
       [0.99046284, 0.00953716],
       [0.24530671, 0.75469329],
       [0.86362667, 0.13637333]])

In [141]:
training_result

array([0, 0, 0, ..., 0, 1, 0])

In [142]:
y_train

array([0, 0, 0, ..., 0, 1, 0])

In [143]:
round((training_result == y_train).mean(), 2)

0.9

In [144]:
y_pred = model.predict(X_val)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [145]:
y_val

array([0, 0, 1, ..., 0, 0, 1])

In [146]:
round((y_pred == y_val).mean(), 2)

0.9

In [147]:
y_pred.shape

(9042,)

In [148]:
X_val.shape

(9042, 47)

## Question 5

## Question 6